# ITM-390: ASD Screening in Children - Final Evaluation & Error Analysis
**Group 10** · American University of Phnom Penh · Advisor: Kuntha PIN  
*Comparative Analysis of Machine Learning Models for Autism Spectrum Disorder Screening in Children*

### Notebook Objectives:
1. Evaluate tuned models once on the untouched **held-out test set (20% split, N=58)**.
2. Compute **1,000 non-parametric bootstrap 95% Confidence Intervals**.
3. Plot overlaid **ROC Curves**, **Precision-Recall Curves**, and **Confusion Matrices**.
4. Perform **Decision Threshold Optimization** for high-sensitivity screening.
5. Conduct **McNemar's Statistical Significance Tests** between algorithm pairs.
6. Perform **Error Analysis** on missed cases (False Negatives).
7. Formulate scientific conclusions regarding the research question and hypothesis.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils import format_markdown_table
print('Evaluation environment loaded.')

## 1. Held-Out Test Set Performance & Bootstrap 95% CIs

In [ ]:
metric_dir = project_root / 'results' / 'metrics'
test_df = pd.read_csv(metric_dir / 'test_set_results.csv')
ci_df = pd.read_csv(metric_dir / 'bootstrap_confidence_intervals.csv')

print('### Held-Out Test Set Performance:')
print(format_markdown_table(test_df))
print('\n### 95% Bootstrap Confidence Intervals:')
print(format_markdown_table(ci_df))

## 2. Publication Visualizations (ROC, PR, Confusion Matrices)

In [ ]:
fig_dir = project_root / 'results' / 'figures'
display(Image(filename=str(fig_dir / 'roc_curves.png')))
display(Image(filename=str(fig_dir / 'confusion_matrices.png')))

## 3. Decision Threshold Analysis (Screening Sensitivity vs Specificity)

In [ ]:
display(Image(filename=str(fig_dir / 'threshold_analysis.png')))

> **Clinical Recommendation:** For an early pediatric screening tool, missed cases (false negatives) are unacceptable. Lowering the decision threshold to ~0.35-0.40 guarantees 100% sensitivity across models while maintaining acceptable specificity.

## 4. Pairwise McNemar's Statistical Significance Tests

In [ ]:
mcnemar_df = pd.read_csv(metric_dir / 'mcnemar_tests.csv')
print(format_markdown_table(mcnemar_df))

## 5. Granular Error Analysis: False Negatives Profiling

In [ ]:
fn_path = metric_dir / 'false_negatives_error_analysis.csv'
if fn_path.exists():
    fn_df = pd.read_csv(fn_path)
    print(f'Total children missed by any model: {len(fn_df)}')
    display(fn_df[['age', 'gender', 'ethnicity', 'jaundice', 'family_asd', 'pred_Logistic Regression', 'pred_K-Nearest Neighbors', 'pred_Random Forest', 'pred_Voting Ensemble']])

## 6. Answers to Research Questions & Hypothesis

- **Research Question:** How do Logistic Regression, KNN, and Random Forest compare in screening for ASD among children?
  - **Logistic Regression & Soft Voting Ensemble** achieved the highest screening efficacy with **100% Recall (0 False Negatives)** on the held-out test set.
  - **Random Forest** achieved the highest overall accuracy (94.83%) and ROC-AUC (0.9976), but missed 1 positive case (Recall = 96.43%).
  - **K-Nearest Neighbors** achieved 96.43% Recall and 82.76% accuracy.

- **Hypothesis:** *The three models will produce different screening performance because they learn decision boundaries in different ways.*
  - **Supported:** McNemar's test confirmed statistically significant differences between linear/ensemble decision boundaries ($p = 0.0159 < 0.05$ between LR and Random Forest, and $p = 0.0233 < 0.05$ between KNN and Random Forest).